# quends — Package Overview

**quends** (Quantified Uncertainty for ENsemble Data Streams) is a scientific Python package designed for uncertainty quantification (UQ) of plasma gyrokinetics simulations.

Plasma turbulence codes such as [GX](https://gx.readthedocs.io) and [Gene](https://genecode.org) produce time-series output — heat flux, particle flux, and other transport quantities — that must be analysed statistically to extract steady-state mean values and confidence intervals.  `quends` automates that workflow:

1. **Load** simulation output into a `DataStream` or `Ensemble`.
2. **Trim** the transient initialisation phase to isolate steady state.
3. **Compute** statistics (mean, standard error, confidence interval) accounting for temporal autocorrelation via effective sample size (ESS).
4. **Export** results as formatted tables or JSON.

This notebook provides a high-level orientation.  The remaining tutorials walk through each workflow step in detail.

In [ ]:
import quends as qnds

# Confirm the package is importable and show the version
print(f"quends version: {qnds.__version__}")

## Architecture Overview

### Core data structures

| Class | Role |
|-------|------|
| `DataStream` | Wraps a single simulation time-series (a pandas `DataFrame`). Provides stationarity testing, ESS, sliding-window statistics, and cumulative statistics. |
| `Ensemble` | Wraps a collection of `DataStream` objects representing repeat runs or parameter scans. Provides member-wise trimming and three ensemble-averaging techniques. |

### I/O helpers (top-level `qnds.*`)

| Function | Description |
|----------|-------------|
| `qnds.from_csv(path)` | Load a CSV file into a `DataStream`. |
| `qnds.from_gx(path, variables=[...])` | Load GX NetCDF output, selecting named variables. |
| `qnds.from_dict(...)` | Construct a `DataStream` from a plain Python `dict`. |
| `qnds.from_numpy(...)` | Construct a `DataStream` from NumPy arrays. |

### Analysis helpers (top-level `qnds.*`)

| Class | Description |
|-------|-------------|
| `qnds.Plotter()` | Produce trace plots, steady-state visualisations, and ACF plots. |
| `qnds.Exporter()` | Display statistics as formatted DataFrames or JSON. |
| `qnds.Ensemble(list_of_datastreams)` | Build an ensemble from a list of `DataStream` objects. |

### Trim strategies (in `quends.base.trim`)

Trimming is the process of discarding the non-stationary warm-up portion of a time series.  All strategies operate with a sliding window and return a start index beyond which the signal is considered stationary.

In [ ]:
# All available trim strategy classes and their factory identifiers
from quends.base.trim import (
    QuantileTrimStrategy,
    NoiseThresholdTrimStrategy,
    RollingVarianceThresholdTrimStrategy,
    SelfConsistentTrimStrategy,
    IQRTrimStrategy,
    TrimDataStreamOperation,
    build_trim_strategy,
)

strategy_summary = [
    ("QuantileTrimStrategy",              "std",             "Discards points whose value falls more than k*MAD (or std) from the rolling median/mean."),
    ("NoiseThresholdTrimStrategy",        "threshold",       "Trims until the rolling std drops below an absolute noise threshold."),
    ("RollingVarianceThresholdTrimStrategy", "rolling_variance", "Trims until the rolling variance stabilises below a relative threshold."),
    ("SelfConsistentTrimStrategy",        "self_consistent", "Iteratively trims until the mean of the trimmed segment stops changing."),
    ("IQRTrimStrategy",                   "iqr",             "Uses the interquartile range to identify and discard outlier-heavy transient regions."),
]

print(f"{'Class':<42} {'factory method':<20} Description")
print("-" * 110)
for cls, method, desc in strategy_summary:
    print(f"{cls:<42} {method:<20} {desc}")

In [ ]:
# The build_trim_strategy factory lets you select a strategy by name string
help(build_trim_strategy)

## Trimming Philosophy — The Explicit Strategy-Operation Pattern

All trimming in `quends` follows a two-object pattern:

1. **Strategy** — encapsulates *how* to decide where the steady state begins.
2. **Operation** (`TrimDataStreamOperation`) — applies the strategy to a specific `DataStream` and column.

This separation keeps the logic testable and composable.  You can swap strategies without changing the application code, and you can inspect or serialise the strategy object independently.

> **Never call `ds.trim()` directly.**  The public DataStream trim API is intentionally not exposed at the instance level to encourage explicit strategy selection.

In [ ]:
# Canonical trim pattern — copy this template into any analysis notebook
from quends.base.trim import QuantileTrimStrategy, TrimDataStreamOperation

# 1. Choose and configure a strategy
strategy = QuantileTrimStrategy(window_size=20, robust=True)

# 2. Wrap it in the operation object
op = TrimDataStreamOperation(strategy=strategy)

# 3. Apply to a DataStream — uncomment once `ds` is loaded
# trimmed = op(ds, column_name="HeatFlux_st")

print("Strategy object:", strategy)
print("Operation object:", op)

## Tutorial Notebook Guide

The full tutorial series covers:

| Notebook | Title | Topic |
|----------|-------|-------|
| **01** | Package Overview (this notebook) | Architecture, trim strategy catalogue |
| **02** | Single DataStream Analysis | Loading, exploring, trimming, and computing statistics for a single run |
| **03** | Ensemble — Technique 0 | Average-then-Analyse: pool all members into one combined stream |
| **04** | Ensemble — Technique 1 | Pooled-Block: block-average each member, then pool the blocks |
| **05** | Ensemble — Technique 2 | Member-wise inverse-variance weighted mean |
| **06** | Trim Strategy Comparison | Side-by-side comparison of all five strategies on the same data |
| **07** | GX NetCDF Loading | Using `qnds.from_gx()` with variable selection |
| **08** | Visualisation Gallery | Full `Plotter` API: trace plots, steady-state overlays, ACF plots |
| **09** | Exporting Results | Formatting and saving statistics tables with `Exporter` |

Start with **02** for a hands-on introduction, or jump directly to the ensemble notebooks (**03–05**) if you are already familiar with single-stream analysis.